# C14 — Class-Balanced CE (β=0.99) + Label Smoothing (ε=0.1) + sqrt_rw

**Tier-1 combo #4 — запускать после C12 и C13.**

Объединяет class rebalancing и label smoothing на оригинальном тексте:
- CBCE β=0.99 (c04): Macro-F1 **0.621**, worst gap 0.388, city-swap flip **0.058**
- LS ε=0.1 (c03): Macro-F1 **0.623**, worst gap 0.318, city-swap flip 0.089

**Гипотеза:** LS смягчит агрессивный CBCE и улучшит gap, сохранив низкий flip и высокий F1.

**Цель:** F1 ≥ 0.620, worst gap ≤ 0.30, flip ≤ 0.07.

**После обучения:** добавить модель в `c10_challenger_city_swap_eval.ipynb`.

**Outputs**
- Model: `notebooks/models/challengers/class_balanced_ls_eps01_beta099_2ep/`
- Summary CSV: `figures/challengers/c14_class_balanced_label_smoothing_summary.csv`
- Results: `notebooks/results/challenger_training/c14_class_balanced_label_smoothing/`

In [1]:
import json
import random

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CWD = Path.cwd()
NOTEBOOKS_DIR = CWD.parent if CWD.name == "challengers" else CWD
PROJECT_ROOT = NOTEBOOKS_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = NOTEBOOKS_DIR / "models" / "challengers"
FIGURES_DIR = PROJECT_ROOT / "figures" / "challengers"
RESULTS_DIR = NOTEBOOKS_DIR / "results" / "challenger_training" / "c14_class_balanced_label_smoothing"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "class_balanced_ls_eps01_beta099_2ep"
BETA = 0.99
SMOOTHING = 0.10
EPOCHS = 2
SUMMARY_CSV = FIGURES_DIR / "c14_class_balanced_label_smoothing_summary.csv"

print(f"CWD: {CWD}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR exists: {(DATA_DIR / 'train.csv').exists()}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")

CWD: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/challengers
PROJECT_ROOT: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
DATA_DIR exists: True
MODELS_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/challengers
FIGURES_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/challengers
RESULTS_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/challenger_training/c14_class_balanced_label_smoothing


In [3]:
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_val = pd.read_csv(DATA_DIR / "val.csv")
df_test = pd.read_csv(DATA_DIR / "test.csv")

mapping = pd.read_csv(DATA_DIR / "label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_val, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])
num_labels = len(le.classes_)

city_counts = df_train["city_group"].value_counts()
raw_w = 1.0 / np.sqrt(city_counts)
city_weight_map = (raw_w / raw_w.mean()).to_dict()
df_train["sample_weight"] = df_train["city_group"].map(city_weight_map).astype(float)

class_counts = df_train["y"].value_counts().sort_index().to_numpy()
print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print(f"Labels: {num_labels}, City groups: {len(city_counts)}")
print(f"Class counts: {class_counts.tolist()}")

Train: 16530, Val: 5510, Test: 5510
Labels: 9, City groups: 41
Class counts: [1546, 2913, 2129, 2006, 1029, 3308, 1016, 1982, 601]


In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize(batch):
    return tokenizer(
        batch["resume_text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )


train_ds = Dataset.from_pandas(df_train[["resume_text", "y", "sample_weight"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(df_val[["resume_text", "y"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(df_test[["resume_text", "y"]]).map(tokenize, batched=True)

train_ds = train_ds.rename_column("y", "labels")
val_ds = val_ds.rename_column("y", "labels")
test_ds = test_ds.rename_column("y", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 5510/5510 [00:47<00:00, 117.11 examples/s]


In [5]:
def effective_num_weights(counts, beta):
    counts = np.asarray(counts, dtype=np.float64)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / np.clip(effective_num, 1e-12, None)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


class_weights = effective_num_weights(class_counts, BETA)
print(f"Class-balanced weights (beta={BETA}): {class_weights.tolist()}")


class ClassBalancedLabelSmoothingTrainer(Trainer):
    def __init__(self, *args, class_weights, smoothing=0.1, num_classes=9, beta=0.99, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.smoothing = smoothing
        self.num_classes = num_classes
        self.beta = beta
        print(f"Class-Balanced + Label Smoothing: beta={beta}, epsilon={smoothing}")

    def label_smooth_loss(self, logits, labels):
        log_probs = F.log_softmax(logits, dim=-1)
        smooth_targets = torch.full_like(log_probs, self.smoothing / (self.num_classes - 1))
        smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - self.smoothing)
        return -(smooth_targets * log_probs).sum(dim=-1)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        sample_weight = inputs.pop("sample_weight", None)
        labels = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits

        loss_per_sample = self.label_smooth_loss(logits, labels)
        class_weights = self.class_weights.to(logits.device)
        loss_per_sample = loss_per_sample * class_weights[labels]

        if sample_weight is not None:
            loss_per_sample = loss_per_sample * sample_weight.to(loss_per_sample.dtype)

        loss = loss_per_sample.mean()
        return (loss, outputs) if return_outputs else loss


def compute_metrics(prediction_output):
    preds = np.argmax(prediction_output.predictions, axis=1)
    return {
        "accuracy": accuracy_score(prediction_output.label_ids, preds),
        "macro_f1": f1_score(prediction_output.label_ids, preds, average="macro"),
    }


def ovr_rates(df, group_col, num_classes):
    groups = sorted(df[group_col].dropna().unique())
    tpr = np.zeros((len(groups), num_classes))
    support = np.zeros((len(groups), num_classes))
    for gi, group_name in enumerate(groups):
        dg = df[df[group_col] == group_name]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(num_classes):
            positive_mask = yt == c
            tp = np.sum((yp == c) & positive_mask)
            fn = np.sum((yp != c) & positive_mask)
            support[gi, c] = positive_mask.sum()
            tpr[gi, c] = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    return tpr, support


def robust_gaps(tpr, support, min_support=30):
    gaps = []
    for c in range(tpr.shape[1]):
        col = tpr[support[:, c] >= min_support, c]
        col = col[~np.isnan(col)]
        gaps.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    gaps = np.array(gaps)
    valid = gaps[~np.isnan(gaps)]
    return (valid.max() if len(valid) else np.nan, valid.mean() if len(valid) else np.nan)

Class-balanced weights (beta=0.99): [0.9997273683547974, 0.999727189540863, 0.999727189540863, 0.999727189540863, 0.9997594356536865, 0.999727189540863, 0.9997639656066895, 0.999727189540863, 1.0021132230758667]


In [6]:
save_dir = MODELS_DIR / MODEL_NAME
checkpoint_dir = save_dir / "checkpoints"


def find_latest_checkpoint(ckpt_dir: Path):
    if not ckpt_dir.exists():
        return None
    ckpts = sorted(
        ckpt_dir.glob("checkpoint-*"),
        key=lambda p: int(p.name.rsplit("-", 1)[-1]),
    )
    return str(ckpts[-1]) if ckpts else None


resume_from = find_latest_checkpoint(checkpoint_dir)

print("=" * 80)
print(f"Training {MODEL_NAME}: beta={BETA}, epsilon={SMOOTHING}, epochs={EPOCHS}")
if resume_from:
    print(f"Resuming from checkpoint: {resume_from}")
else:
    print("No checkpoint found — training from scratch")

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
args = TrainingArguments(
    output_dir=str(checkpoint_dir),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)

trainer = ClassBalancedLabelSmoothingTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    smoothing=SMOOTHING,
    num_classes=num_labels,
    beta=BETA,
)

trainer.train(resume_from_checkpoint=resume_from)
pred_output = trainer.predict(test_ds)
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

acc = float(accuracy_score(y_true, y_pred))
macro_f1 = float(f1_score(y_true, y_pred, average="macro"))

df_eval = df_test.copy()
df_eval["y_true"] = y_true
df_eval["y_pred"] = y_pred
tpr, support = ovr_rates(df_eval, "city_group", num_labels)
worst_gap, macro_gap = robust_gaps(tpr, support, min_support=30)

save_dir.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(save_dir, safe_serialization=True)
tokenizer.save_pretrained(save_dir)
joblib.dump(le, save_dir / "label_encoder.joblib")

training_config = {
    "method": "Class-Balanced CE + Label Smoothing + sqrt_rw",
    "model_name": MODEL_NAME,
    "beta": BETA,
    "smoothing": SMOOTHING,
    "epochs": EPOCHS,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "accuracy": acc,
    "macro_f1": macro_f1,
    "tpr_gap_worst_robust": float(worst_gap),
    "tpr_gap_macro_robust": float(macro_gap),
}
with open(save_dir / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2, ensure_ascii=False)

summary_row = {
    "model_name": MODEL_NAME,
    "method": "Class-Balanced CE + Label Smoothing + sqrt_rw",
    "beta": BETA,
    "smoothing": SMOOTHING,
    "epochs": EPOCHS,
    "accuracy": acc,
    "macro_f1": macro_f1,
    "worst_gap": float(worst_gap),
    "macro_gap": float(macro_gap),
    "model_dir": str(save_dir.relative_to(PROJECT_ROOT)),
}
summary_df = pd.DataFrame([summary_row])
summary_df.to_csv(SUMMARY_CSV, index=False)
summary_df.to_csv(RESULTS_DIR / "training_summary.csv", index=False)

with open(RESULTS_DIR / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2, ensure_ascii=False)

pred_df = df_eval[["city_group", "label", "supercategory", "y_true", "y_pred"]].copy()
pred_df.to_csv(RESULTS_DIR / "predictions_test.csv", index=False)

print(f"\nTEST: Acc={acc:.4f}, Macro-F1={macro_f1:.4f}")
print(f"FAIRNESS (robust): worst={worst_gap:.4f}, macro={macro_gap:.4f}")
print(f"Model saved to: {save_dir}")
print(f"Summary CSV: {SUMMARY_CSV}")
print(f"Results dir: {RESULTS_DIR}")
summary_df

Training class_balanced_ls_eps01_beta099_2ep: beta=0.99, epsilon=0.1, epochs=2
No checkpoint found — training from scratch


Loading weights: 100%|██████████| 199/199 [00:06<00:00, 30.26it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those par

Class-Balanced + Label Smoothing: beta=0.99, epsilon=0.1


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.577196,1.379924,0.584755,0.616722
2,0.571896,1.331882,0.613430,0.633834


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.39s/it]
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.La

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.26it/s]



TEST: Acc=0.6031, Macro-F1=0.6172
FAIRNESS (robust): worst=0.3412, macro=0.1237
Model saved to: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/challengers/class_balanced_ls_eps01_beta099_2ep
Summary CSV: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/challengers/c14_class_balanced_label_smoothing_summary.csv
Results dir: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/challenger_training/c14_class_balanced_label_smoothing


,model_name,method,beta,smoothing,epochs,accuracy,macro_f1,worst_gap,macro_gap,model_dir
0,class_balanced_ls_eps01_beta099_2ep,Class-Balanced CE + Label Smoothing + sqrt_rw,0.99,0.1,2,0.603085,0.617245,0.341176,0.123714,notebooks/models/challengers/class_balanced_ls...
